In [11]:
import osmnx as ox
import geopandas as gpd
import pandas as pd


In [7]:
def get_building_info_by_coordinates(lat, lon, radius=50):

    center_point = (lat, lon)
    tags = {"building": True} 

    gdf = ox.features.features_from_point(center_point=center_point, tags=tags, dist=radius)

    if not gdf.empty:
        
        for col in ['height', 'building:material', 'seismic:resistance']:
            if col not in gdf.columns:
                gdf[col] = None  

        
        gdf['height_m'] = gdf['height'].str.extract(r'(\d+)', expand=False).astype(float)

        
        material_counts = gdf['building:material'].value_counts()
        print("\nBuilding materials distribution statistics:")
        print(material_counts)

       
        gdf['seismic:resistance'] = gdf['seismic:resistance'].fillna("Unknown")  

        return gdf
    else:
        print("No buildings found.")
        return gpd.GeoDataFrame() 

In [18]:
#latitude = 35.6225  
#longitude = -117.6709
radius = 100  
# or download_images(35.6225, -117.6709)
latitude = 48.858372  
longitude = 2.294481 
result = get_building_info_by_coordinates(latitude, longitude, radius)


if not result.empty:
    
    print("\nBuilding Information:")
    print(result[["building", "height", "height_m", "building:material", "seismic:resistance", "geometry"]])
else:
    print("No buildings found.")


Building materials distribution statistics:
iron    1
Name: building:material, dtype: int64

Building Information:
                  building height  height_m building:material  \
element id                                                      
way     5013364      tower    324     324.0              iron   
        69034130       yes    NaN       NaN               NaN   
        69034165       yes    NaN       NaN               NaN   
        335101041     roof    NaN       NaN               NaN   
        335101043      yes      3       3.0               NaN   
        335260394      yes    NaN       NaN               NaN   
        449952613      yes    NaN       NaN               NaN   

                  seismic:resistance  \
element id                             
way     5013364              Unknown   
        69034130             Unknown   
        69034165             Unknown   
        335101041            Unknown   
        335101043            Unknown   
        335260394 

In [19]:
def describe_buildings(gdf, radius=50):
    description = []

    total = len(gdf)
    description.append(f"A total of {total} buildings were found within a {radius} meter radius")

    # Building types
    if 'building' in gdf.columns:
        building_types = gdf['building'].value_counts()
        if not building_types.empty:
            types_desc = ", ".join(f"{btype} ({count})" for btype, count in building_types.items())
            description.append(f", including types such as: {types_desc}")

    # Heights
    if 'height_m' in gdf.columns:
        known_heights = gdf[gdf['height_m'].notna()]
        if not known_heights.empty:
            min_h = known_heights['height_m'].min()
            max_h = known_heights['height_m'].max()
            if min_h == max_h:
                description.append(f". Building height is approximately {min_h:.1f} meters")
            else:
                description.append(f". Building heights range from {min_h:.1f} to {max_h:.1f} meters")

    # Materials
    if 'building:material' in gdf.columns:
        material_counts = gdf['building:material'].dropna().value_counts()
        if not material_counts.empty:
            mats = ", ".join(f"{mat} ({cnt})" for mat, cnt in material_counts.items())
            description.append(f". Building materials include: {mats}")

    # Seismic resistance
    if 'seismic:resistance' in gdf.columns:
        seismic_known = gdf[gdf['seismic:resistance'].notna() & (gdf['seismic:resistance'] != 'Unknown')]
        if not seismic_known.empty:
            types = seismic_known['seismic:resistance'].value_counts()
            desc = ", ".join(f"{k} ({v})" for k, v in types.items())
            description.append(f". Some buildings have seismic resistance information: {desc}")
        else:
            description.append(". No seismic resistance information is available for these buildings")

    description.append(".")
    return "".join(description)


In [20]:
summary = describe_buildings(result, radius)
print(summary)


A total of 7 buildings were found within a 100 meter radius, including types such as: yes (5), tower (1), roof (1). Building heights range from 3.0 to 324.0 meters. Building materials include: iron (1). No seismic resistance information is available for these buildings.
